# Modelos: Regresión logística, Naive Bayes y Gradient Boosting
Entreno mis modelos con el preprocesamiento común (`src/utils/modelado.py`) y guardo los resultados en `resultados/`, para compararlos al final con los del resto del grupo.

## 0. Modelos probados y justificación

| Modelo | Tipo | Por qué en este problema |
|---|---|---|
| Regresión logística | Base | En el EDA, cada variable sube o baja un poco el riesgo, siempre en la misma dirección, y casi no se relacionan entre sí. Es una suma de efectos pequeños, justo lo que modela. Además, sus coeficientes se pueden explicar al médico |
| Naive Bayes | Base | Supone que las variables son independientes dentro de cada clase, y en el EDA que realicé apenas se relacionan entre sí |
| HistGradientBoosting | Ensemble | Suma árboles pequeños que corrigen los errores de los anteriores: puede captar efectos no lineales que la logística no ve. Es la versión mejorada de Gradient Boosting y admite `class_weight` para poder comparar con los 3 balanceos |

## 1. Importo librerías y cargo los datos con preprocesamiento

In [ ]:
from utils import modelado as mod, constantes as cte
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import HistGradientBoostingClassifier

%matplotlib inline

# misma partición 80/20 estratificada que el resto del grupo; X_test no se usa en este notebook
X_train, X_test, y_train, y_test = mod.cargar_datos()
print(f"Train: {X_train.shape[0]} pacientes y {X_train.shape[1]} variables | Test: {X_test.shape[0]} pacientes")
print("Proporción de clases en train:", y_train.value_counts(normalize=True).sort_index().round(2).to_dict())

Train: 1803 pacientes y 30 variables | Test: 451 pacientes
Proporción de clases en train: {0: 0.5, 1: 0.3, 2: 0.2}


## 2. Modelos por defecto con cada balanceo

Como acordamos en nuestra última reunión y siguiendo el pipeline de `modelado.py` probamos los 3 balanceos de la pipeline con los parámetros por defecto. Naive Bayes no tiene un parámetro de semilla, por lo que no se rellena.

In [12]:
modelos = {
    "logistica": LogisticRegression(random_state=cte.semilla),
    "naive_bayes": GaussianNB(),
    "hist_gradient_boosting": HistGradientBoostingClassifier(random_state=cte.semilla),
}

balanceos = [None, "class_weight", "smotenc"]
filas = []
pipeline = {}
busqueda_por_defecto = {}
for nombre, modelo in modelos.items():
    for balanceo in balanceos:
        etiqueta = f"{nombre}_{balanceo or "sin_balanceo"}"
        try:
            pipeline[etiqueta] = mod.crear_pipeline(modelo, balanceo)
        except ValueError:
            print(f"El modelo {nombre} no admite {balanceo} como parámetro. Se omite.")
            continue
        busqueda_por_defecto[etiqueta] = mod.crear_busqueda(pipeline[etiqueta], {}).fit(X_train, y_train)
        
por_defecto = pd.concat([mod.tabla_resultados(b, etiqueta) for etiqueta, b in busqueda_por_defecto.items()])

# train frente a validación: el sobreajuste se ve en la diferencia, sin tocar el test
columnas = ["f1_macro_train", "f1_macro_val", "f1_macro_val_std", "recall_alto_val",
            "precision_alto_val", "diferencia_train_val"]
por_defecto[columnas].sort_values("f1_macro_val", ascending=False).round(3)

El modelo naive_bayes no admite class_weight como parámetro. Se omite.


,f1_macro_train,f1_macro_val,f1_macro_val_std,recall_alto_val,precision_alto_val,diferencia_train_val
modelo,,,,,,
logistica_class_weight,0.673,0.648,0.037,0.737,0.636,0.025
logistica_sin_balanceo,0.674,0.645,0.026,0.670,0.693,0.029
naive_bayes_sin_balanceo,0.664,0.641,0.039,0.684,0.653,0.023
logistica_smotenc,0.666,0.637,0.031,0.753,0.622,0.030
hist_gradient_boosting_sin_balanceo,1.000,0.629,0.027,0.634,0.680,0.371
hist_gradient_boosting_smotenc,1.000,0.624,0.030,0.687,0.642,0.376
hist_gradient_boosting_class_weight,1.000,0.621,0.033,0.657,0.637,0.379
naive_bayes_smotenc,0.633,0.602,0.029,0.734,0.580,0.031


**Conclusión.**

- **La regresión logística y Naive Bayes apenas sobreajustan:** entre 0,02 y 0,03 de diferencia entre train y validación.
- **HistGradientBoosting memoriza el train:** F1 de 1,0 en train frente a 0,62-0,63 en validación, con los tres balanceos. Necesita regularización, y es lo que busca su rejilla (apartado 5).
- **Balancear sube el recall de riesgo alto:** en la logística pasa de 0,67 a 0,74-0,75 con `class_weight` o SMOTENC, sin bajar el F1 macro.
- No elijo aquí el balanceo: con los valores por defecto la comparación no es justa (por ejemplo, HistGradientBoosting memoriza con los tres). Lo decido en cada apartado, con los hiperparámetros ajustados.

A continuación procedo a ajustar los hiperparámetros de cada modelo

In [26]:
# los mejores parámetros son un texto largo: así pandas no lo corta con "..."
pd.set_option("display.max_colwidth", None)


def ajustar_modelo(nombre, modelo, parametros, balanceos=balanceos):
    """Ajusta la rejilla con cada balanceo y devuelve (búsquedas, tabla comparativa).

    nombre: prefijo de las filas, p. ej. "logistica" -> "logistica_smotenc".
    parametros: rejilla con el prefijo "modelo__", p. ej. {"modelo__C": [0.1, 1]}.
    """
    busquedas, filas = {}, []
    for balanceo in balanceos:
        # misma etiqueta que en el apartado 2, para comparar con los valores por defecto
        etiqueta = f"{nombre}_{balanceo or 'sin_balanceo'}"
        try:
            pipeline = mod.crear_pipeline(modelo, balanceo)
        except ValueError:
            # p. ej. GaussianNB no admite class_weight: se salta ese balanceo
            print(f"El modelo {nombre} no admite {balanceo} como parámetro. Se omite.")
            continue
        # crear_busqueda pone la validación, la semilla y las métricas comunes del grupo
        busquedas[etiqueta] = mod.crear_busqueda(pipeline, parametros).fit(X_train, y_train)
        filas.append(mod.tabla_resultados(busquedas[etiqueta], etiqueta))

    # mismas columnas que en el apartado 2 y, además, la mejor combinación de cada balanceo
    tabla = pd.concat(filas)[columnas + ["mejores_parametros"]]
    return busquedas, tabla.sort_values("f1_macro_val", ascending=False)

## 3. Regresión logística

Hiperparámetros:

- `C`: *inversa de la fuerza de la regularización*. Un valor alto indica una regularización débil o nula, permitiendo un modelo más complejo, mientras que uno pequeño lo simplifica. Pruebo de 0,001 a 100.
- `l1_ratio`: *tipo de regularización*. Un valor 0 aplica una penalización L2 pura (Ridge), que mantiene todos los coeficientes pero los reduce suavemente. Mientras que el valor 1 hace lo propio con L1 (Lasso), que tiende a producir modelos dispersos a llevar algunos coeficientes a cero mediante selección de características.
- Selecciono `saga` como solver, dado que admite L1, L2 y Elastic-Net. `liblinear` soporta L1 y L2 pero es paea clasificación binaria y el valor por predeterinado `lbfgs`solo soporta L2.
- `max_iter` aunque no es un hiperparámetro es el límite de pasos del optimizaor hasta que el modelo converge, conviene tenerlo alto para optimizar aunque depende del solver.

In [27]:
parametros_logistica = {
    "modelo__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "modelo__l1_ratio": [0, 0.5, 1],
}

logistica = LogisticRegression(solver="saga", max_iter=5000, random_state=cte.semilla)

busquedas_logistica, tabla_logistica = ajustar_modelo("logistica", logistica, parametros_logistica)
tabla_logistica.round(3)

,f1_macro_val,f1_macro_val_std,recall_alto_val,recall_alto_val_std,precision_alto_val,f1_macro_train,diferencia_train_val,mejores_parametros,mejores_parametros
modelo,,,,,,,,,
logistica_class_weight,0.651,0.034,0.748,0.059,0.631,0.675,0.024,"{'modelo__C': 0.1, 'modelo__l1_ratio': 0}","{'modelo__C': 0.1, 'modelo__l1_ratio': 0}"
logistica_sin_balanceo,0.647,0.027,0.670,0.045,0.691,0.671,0.025,"{'modelo__C': 0.1, 'modelo__l1_ratio': 0}","{'modelo__C': 0.1, 'modelo__l1_ratio': 0}"
logistica_smotenc,0.637,0.029,0.753,0.067,0.622,0.667,0.030,"{'modelo__C': 1, 'modelo__l1_ratio': 0}","{'modelo__C': 1, 'modelo__l1_ratio': 0}"


In [29]:
tabla_logistica.loc["logistica_class_weight", "mejores_parametros"]

mejores_parametros    {'modelo__C': 0.1, 'modelo__l1_ratio': 0}
mejores_parametros    {'modelo__C': 0.1, 'modelo__l1_ratio': 0}
Name: logistica_class_weight, dtype: str

In [30]:
tabla_logistica.columns

Index(['f1_macro_val', 'f1_macro_val_std', 'recall_alto_val',
       'recall_alto_val_std', 'precision_alto_val', 'f1_macro_train',
       'diferencia_train_val', 'mejores_parametros', 'mejores_parametros'],
      dtype='str')

In [35]:
columnas_log = {}
for metrica in ("f1_macro", "recall_alto"):
    for balanceo, b in busquedas_logistica.items():
        columnas_log[(metrica, str(balanceo))] = [b.cv_results_[f"split{k}_test_{metrica}"][b.best_index_]
                                                for k in range(cte.n_folds)]
tabla_log = pd.DataFrame(columnas_log)
tabla_log.index.name = "fold"

display(tabla_log)

f1_macro                                           \
     logistica_sin_balanceo logistica_class_weight logistica_smotenc   
fold                                                                   
0                  0.667446               0.677678          0.659691   
1                  0.684426               0.661945          0.641316   
2                  0.615271               0.611026          0.607808   
3                  0.648133               0.693264          0.674933   
4                  0.618363               0.612341          0.599763   

                recall_alto                                           
     logistica_sin_balanceo logistica_class_weight logistica_smotenc  
fold                                                                  
0                  0.680556               0.722222          0.736111  
1                  0.712329               0.767123          0.753425  
2                  0.583333               0.666667          0.652778  
3                  0.694444               0.847222          0.861111  
4                  0.680556               0.736111          0.763889

In [37]:
busqueda_logistica = busquedas_logistica["logistica_class_weight"]
print("Mejores parámetros:", busqueda_logistica.best_params_)

Mejores parámetros: {'modelo__C': 0.1, 'modelo__l1_ratio': 0}
